# Step 4: RQ2 Prompt Token Counts

This notebook estimates input prompt length for the RQ2 zero-shot and few-shot prompts. It counts `system_message + user_message` tokens for each sampled instance under Top-4 and Top-24 settings.

The notebook only reads existing data and few-shot example files. It does not call any LLM APIs and does not modify existing experiment results.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

PROJECT_ROOT = Path.cwd()
COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_ClubLending")


def find_local_project_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Data" / "ModifiedData").exists() and (candidate / "Result").exists():
            return candidate
    return start.parent if start.name == "Code" else start


BASE_DIR = COLAB_PROJECT_ROOT if COLAB_PROJECT_ROOT.exists() else find_local_project_root(PROJECT_ROOT)
MODIFIED_DATA_DIR = BASE_DIR / "Data" / "ModifiedData"
LLM_DIR = BASE_DIR / "Result" / "LLM"
OUTPUT_DIR = LLM_DIR / "RQ2_Prompt_Token_Counts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_DATA_PATH = MODIFIED_DATA_DIR / "LC_split.pkl"

print(f"Base directory: {BASE_DIR}")
print(f"LLM directory: {LLM_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


## 2. Tokenizer

Uses `tiktoken` with `cl100k_base`, which is appropriate for GPT-style estimates. For Claude and Gemini, this should be treated as an approximate but consistent prompt-length measure.

In [ ]:
try:
    import tiktoken
except ImportError as exc:
    raise ImportError(
        "Please install tiktoken in this environment first: pip install tiktoken"
    ) from exc

TOKENIZER_NAME = "cl100k_base"
encoder = tiktoken.get_encoding(TOKENIZER_NAME)


def count_tokens(text):
    return len(encoder.encode(str(text)))


print(f"Tokenizer: {TOKENIZER_NAME}")


## 3. Feature Names and Formatting

In [ ]:
FEATURE_TO_NAME_MAP = {
    "home_ownership": "Home Ownership Status",
    "verification_status": "Income Verification Status",
    "purpose": "Loan Purpose",
    "area": "Borrower Area",
    "addr_state": "Borrower State",
    "term": "Number of Payments",
    "grade": "Loan Grade",
    "sub_grade": "Loan Sub Grade",
    "emp_length": "Employment Length",
    "pub_rec_bankruptcies": "Number of Public Bankruptcies",
    "funded_amnt": "Loan Amount",
    "int_rate": "Interest Rate",
    "installment": "Monthly Payment",
    "annual_inc": "Annual Income",
    "dti": "Debt to Income Ratio",
    "delinq_2yrs": "Number of Delinquencies in Past 2 Years",
    "fico_range_low": "Lowest FICO Score",
    "fico_range_high": "Highest FICO Score",
    "inq_last_6mths": "Number of Inquiries in Last 6 Months",
    "open_acc": "Number of Open Accounts",
    "pub_rec": "Number of Derogatory Public Records",
    "revol_bal": "Revolving Balance",
    "revol_util": "Revolving Utilization Rate",
    "total_acc": "Total Number of Accounts",
}

OUTPUT_SCHEMA = {
    "ranked_features": [
        "Feature name 1",
        "Feature name 2",
        "..."
    ]
}


def readable_feature_name(feature):
    return FEATURE_TO_NAME_MAP.get(feature, feature)


def format_feature_value(feature, value):
    if pd.isna(value):
        return "Missing"
    if feature in ["int_rate", "revol_util"]:
        return f"{float(value) * 100:.1f}%"
    if feature == "annual_inc":
        return f"{float(value):,.0f}"
    if isinstance(value, (np.floating, float)):
        return f"{float(value):.4g}"
    return str(value)


def input_table_for_row(row):
    lines = [
        "Feature                                  | Input Value",
        "-" * 72,
    ]
    for feature, value in row.items():
        readable = readable_feature_name(feature)
        lines.append(f"{readable:<40s} | {format_feature_value(feature, value)}")
    return "\n".join(lines)


## 4. Load Existing Data

In [ ]:
if not SPLIT_DATA_PATH.exists():
    raise FileNotFoundError(f"Missing split data file: {SPLIT_DATA_PATH}")

with open(SPLIT_DATA_PATH, "rb") as f:
    X_train, X_test, y_train, y_test = pickle.load(f)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

sample_files = {
    "Logistic Regression": LLM_DIR / "500_samples_logistic.xlsx",
    "XGBoost": LLM_DIR / "500_samples_xgboost_matched.xlsx",
}

few_shot_files = {
    "Logistic Regression": LLM_DIR / "RQ2_Logistic_FewShot_Redesigned" / "few_shot_examples.pkl",
    "XGBoost": LLM_DIR / "RQ2_XGBoost_FewShot_Redesigned" / "few_shot_examples.pkl",
}

sample_dfs = {}
few_shot_examples = {}

for base_model, path in sample_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing sample file for {base_model}: {path}")
    sample_dfs[base_model] = pd.read_excel(path)
    sample_dfs[base_model]["Rowindex"] = sample_dfs[base_model]["Rowindex"].astype(int)

for base_model, path in few_shot_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing few-shot example file for {base_model}: {path}")
    with open(path, "rb") as f:
        few_shot_examples[base_model] = pickle.load(f)

print(f"X_test shape: {X_test.shape}")
for base_model, df in sample_dfs.items():
    print(f"{base_model}: {len(df)} samples")
for base_model, examples in few_shot_examples.items():
    print(f"{base_model}: {len(examples)} few-shot examples")


## 5. Reconstruct RQ2 Prompts

In [ ]:
def prepare_sample(rowindex, X, sample_df):
    rowindex = int(rowindex)
    pred_prob = sample_df.loc[sample_df["Rowindex"].eq(rowindex), "PredProb"].iloc[0]
    return {
        "sample_index": rowindex,
        "predicted_probability": float(pred_prob),
        "input_table": input_table_for_row(X.loc[rowindex]),
        "allowed_readable_features": [readable_feature_name(feature) for feature in X.columns],
    }


def build_rq2_zeroshot_prompt(sample, base_model, top_n):
    system_message = f"""
You are an expert credit risk analyst.

Your task is to infer which borrower input features most likely contributed to the {base_model} model prediction.
Rules:
1. Use ONLY the borrower input values provided below.
2. Do not use attribution values, SHAP values, or a model-provided ranking; none are shown.
3. Return the top {top_n} features from most important to least important.
4. Copy feature names exactly as listed in the provided feature list.
5. Do not invent, rename, merge, or explain features.
6. Return ONLY valid JSON, with no Markdown fences, prose, comments, or explanation.
7. Use exactly this schema:
{json.dumps(OUTPUT_SCHEMA, indent=2)}
""".strip()

    user_message = f"""
PREDICTED DEFAULT PROBABILITY: {sample['predicted_probability']:.4f}

BORROWER INPUT VALUES:
{sample['input_table']}

Task:
Rank the top {top_n} features that most likely contributed to this {base_model} model prediction.
""".strip()

    return system_message, user_message


def format_few_shot_examples(examples):
    blocks = []
    for number, example in enumerate(examples, start=1):
        ranked = "\n".join(
            f"{rank}. {feature}"
            for rank, feature in enumerate(example["ranked_features"], start=1)
        )
        block = f"""
EXAMPLE {number} ({example['selection_role']}):

BORROWER INPUT VALUES:
{example['input_table']}

CORRECT TOP FEATURES:
{ranked}
""".strip()
        blocks.append(block)
    return "\n\n".join(blocks)


def build_rq2_fewshot_prompt(sample, examples, base_model, top_n):
    example_text = format_few_shot_examples(examples)

    if base_model == "XGBoost":
        system_message = f"""
You are an expert credit risk analyst.

You will see two labeled examples: one training example with the highest model-predicted default probability and one with the lowest model-predicted default probability.
Each example shows borrower input values and the correct top XGBoost model features.
Then you will rank the top {top_n} features for a new borrower.
Rules:
1. Use ONLY the target borrower's input values.
2. Do not use attribution values, SHAP values, or a model-provided ranking for the target; none are shown.
3. Return the top {top_n} features from most important to least important.
4. Choose feature names only as listed in the allowed feature list.
6. Do not invent, rename, merge, or explain features.
7. Return ONLY valid JSON, with no Markdown fences, prose, comments, or explanation.
8. Use exactly this schema:
{json.dumps(OUTPUT_SCHEMA, indent=2)}
""".strip()
    else:
        system_message = f"""
You are an expert credit risk analyst.

You will see two labeled examples: one training example with the highest model-predicted default probability and one with the lowest model-predicted default probability.
Each example shows borrower input values and the correct top Logistic Regression model features.
Then you will rank the top {top_n} features for a new borrower.
Rules:
1. Use ONLY the target borrower's input values.
2. Do not use attribution values, SHAP values, or a model-provided ranking for the target; none are shown.
3. Return the top {top_n} features from most important to least important.
4. Choose feature names exactly as listed in the provided feature list.
5. Do not invent, rename, merge, or explain features.
6. Return ONLY valid JSON, with no Markdown fences, prose, comments, or explanation.
7. Use exactly this schema:
{json.dumps(OUTPUT_SCHEMA, indent=2)}
""".strip()

    user_message = f"""
FEW-SHOT EXAMPLES:
{example_text}

TARGET BORROWER PREDICTED DEFAULT PROBABILITY: {sample['predicted_probability']:.4f}

TARGET BORROWER INPUT VALUES:
{sample['input_table']}

Task:
Rank the top {top_n} features that most likely contributed to this {base_model} model prediction.
""".strip()

    return system_message, user_message


## 6. Count Tokens by Sample

In [ ]:
def count_prompt_tokens(system_message, user_message):
    system_tokens = count_tokens(system_message)
    user_tokens = count_tokens(user_message)
    return {
        "SystemTokens": system_tokens,
        "UserTokens": user_tokens,
        "TotalPromptTokens": system_tokens + user_tokens,
    }


rows = []
for base_model in ["Logistic Regression", "XGBoost"]:
    sample_df = sample_dfs[base_model]
    examples = few_shot_examples[base_model]

    for top_n in [4, 24]:
        for condition in ["ZeroShot", "FewShot"]:
            for rowindex in sample_df["Rowindex"].astype(int):
                sample = prepare_sample(rowindex, X_test, sample_df)

                if condition == "ZeroShot":
                    system_message, user_message = build_rq2_zeroshot_prompt(
                        sample,
                        base_model=base_model,
                        top_n=top_n,
                    )
                else:
                    system_message, user_message = build_rq2_fewshot_prompt(
                        sample,
                        examples=examples,
                        base_model=base_model,
                        top_n=top_n,
                    )

                token_counts = count_prompt_tokens(system_message, user_message)
                rows.append({
                    "BaseModel": base_model,
                    "Condition": condition,
                    "TopN": top_n,
                    "Rowindex": rowindex,
                    **token_counts,
                })

prompt_token_detail_df = pd.DataFrame(rows)
display(prompt_token_detail_df.head())
print(f"Rows: {len(prompt_token_detail_df)}")


## 7. Summary Table

In [ ]:
prompt_token_summary_df = (
    prompt_token_detail_df
    .groupby(["BaseModel", "Condition", "TopN"], dropna=False)
    .agg(
        N=("Rowindex", "size"),
        System_mean=("SystemTokens", "mean"),
        User_mean=("UserTokens", "mean"),
        Total_mean=("TotalPromptTokens", "mean"),
        Total_min=("TotalPromptTokens", "min"),
        Total_median=("TotalPromptTokens", "median"),
        Total_max=("TotalPromptTokens", "max"),
    )
    .reset_index()
)

round_cols = ["System_mean", "User_mean", "Total_mean", "Total_min", "Total_median", "Total_max"]
prompt_token_summary_df[round_cols] = prompt_token_summary_df[round_cols].round(1)

display(prompt_token_summary_df)


## 8. Save Outputs

In [ ]:
detail_path = OUTPUT_DIR / "rq2_prompt_token_counts_by_sample.xlsx"
summary_path = OUTPUT_DIR / "rq2_prompt_token_counts_summary.xlsx"

prompt_token_detail_df.to_excel(detail_path, index=False)
prompt_token_summary_df.to_excel(summary_path, index=False)

print(f"Saved detail file: {detail_path}")
print(f"Saved summary file: {summary_path}")


## 9. RQ1 Prompt Token Counts

This section estimates the input prompt length for the redesigned RQ1 translator prompts. RQ1 prompts contain only the model-provided ordered feature list, not borrower feature values.

In [ ]:
def rq1_logistic_sample(rowindex, contribution_df, sample_df, threshold=0.556):
    rowindex = int(rowindex)
    contribution_row = contribution_df.loc[rowindex].copy()
    contribution_table_df = contribution_row.reset_index()
    contribution_table_df.columns = ["Feature", "Contribution"]
    contribution_table_df["AbsContribution"] = contribution_table_df["Contribution"].abs()
    contribution_table_df = contribution_table_df.sort_values("AbsContribution", ascending=False).reset_index(drop=True)
    contribution_table_df["Feature"] = contribution_table_df["Feature"].apply(readable_feature_name)

    sample_row = sample_df[sample_df["Rowindex"].astype(int) == rowindex].iloc[0]
    pred_prob = float(sample_row["PredProb"])
    label = int(sample_row["Label"])

    return {
        "sample_index": rowindex,
        "pred_prob": pred_prob,
        "pred_res": "default" if pred_prob >= threshold else "not_default",
        "true_res": "default" if label == 1 else "not_default",
        "attribution_table": "\n".join(contribution_table_df["Feature"].tolist()),
        "reference_rank": contribution_table_df["Feature"].tolist(),
    }


def rq1_xgboost_sample(rowindex, attribution_df, sample_df, threshold=0.556):
    rowindex = int(rowindex)
    attribution_row = attribution_df[attribution_df["Rowindex"].astype(int) == rowindex].copy()
    attribution_row = attribution_row.sort_values("AbsSHAPValue", ascending=False).reset_index(drop=True)

    sample_row = sample_df[sample_df["Rowindex"].astype(int) == rowindex].iloc[0]
    pred_prob = float(sample_row["PredProb"])
    label = int(sample_row["Label"])

    feature_lines = [readable_feature_name(feature) for feature in attribution_row["Feature"].astype(str).tolist()]

    return {
        "sample_index": rowindex,
        "pred_prob": pred_prob,
        "pred_res": "default" if pred_prob >= threshold else "not_default",
        "true_res": "default" if label == 1 else "not_default",
        "attribution_table": "\n".join(feature_lines),
        "reference_rank": feature_lines,
    }


def build_rq1_prompt(sample, base_model):
    if base_model == "Logistic Regression":
        model_phrase = "a Logistic Regression"
        rule_2 = "Do not reorder features based on feature meaning or credit-risk knowledge."
    else:
        model_phrase = "an XGBoost"
        rule_2 = "Do not reorder features based on attribution values or credit-risk knowledge."

    system_message = f"""
You are given {model_phrase} model-provided feature importance list.

The features are already ordered from most important to least important.

Your task is to reproduce the feature order exactly as provided.
Rules:
1. Copy the feature names in the exact order shown.
2. {rule_2}
3. Do not add, remove, rename, merge, or explain features.
4. Return ONLY valid JSON, with no Markdown fences, prose, comments, or explanation.
5. Use exactly this schema:
{json.dumps(OUTPUT_SCHEMA, indent=2)}
""".strip()

    user_message = f"""
MODEL-PROVIDED FEATURE IMPORTANCE LIST:
The features below are already ordered from most important to least important.

{sample['attribution_table']}

Task:
Return the feature names in exactly the same order as shown above.
""".strip()

    return system_message, user_message


rq1_sample_files = {
    "Logistic Regression": LLM_DIR / "500_samples_logistic.xlsx",
    "XGBoost": LLM_DIR / "500_samples_xgboost_matched.xlsx",
}
rq1_reference_files = {
    "Logistic Regression": BASE_DIR / "Result" / "SHAP" / "Logistic_Contrib.xlsx",
    "XGBoost": BASE_DIR / "Result" / "SHAP_retrained" / "XGBoost_SHAP_Aggregated.xlsx",
}

rq1_sample_dfs = {
    base_model: pd.read_excel(path)
    for base_model, path in rq1_sample_files.items()
}
rq1_sample_dfs["Logistic Regression"]["Rowindex"] = rq1_sample_dfs["Logistic Regression"]["Rowindex"].astype(int)
rq1_sample_dfs["XGBoost"]["Rowindex"] = rq1_sample_dfs["XGBoost"]["Rowindex"].astype(int)

rq1_log_contrib = pd.read_excel(rq1_reference_files["Logistic Regression"], index_col=0)
rq1_log_contrib.index = rq1_log_contrib.index.astype(int)

rq1_xgb_attr = pd.read_excel(rq1_reference_files["XGBoost"])
rq1_xgb_attr["Rowindex"] = rq1_xgb_attr["Rowindex"].astype(int)
if "AbsSHAPValue" not in rq1_xgb_attr.columns:
    if "AbsContribution" in rq1_xgb_attr.columns:
        rq1_xgb_attr["AbsSHAPValue"] = rq1_xgb_attr["AbsContribution"].abs()
    elif "SignedContribution" in rq1_xgb_attr.columns:
        rq1_xgb_attr["AbsSHAPValue"] = rq1_xgb_attr["SignedContribution"].abs()
    else:
        raise KeyError("XGBoost attribution table needs AbsSHAPValue, AbsContribution, or SignedContribution.")

rq1_rows = []
for base_model in ["Logistic Regression", "XGBoost"]:
    sample_df = rq1_sample_dfs[base_model]
    for rowindex in sample_df["Rowindex"].astype(int):
        if base_model == "Logistic Regression":
            sample = rq1_logistic_sample(rowindex, rq1_log_contrib, sample_df)
        else:
            sample = rq1_xgboost_sample(rowindex, rq1_xgb_attr, sample_df)

        system_message, user_message = build_rq1_prompt(sample, base_model)
        token_counts = count_prompt_tokens(system_message, user_message)
        rq1_rows.append({
            "ResearchQuestion": "RQ1",
            "BaseModel": base_model,
            "Rowindex": rowindex,
            **token_counts,
        })

rq1_prompt_token_detail_df = pd.DataFrame(rq1_rows)
rq1_prompt_token_summary_df = (
    rq1_prompt_token_detail_df
    .groupby(["ResearchQuestion", "BaseModel"], dropna=False)
    .agg(
        N=("Rowindex", "size"),
        System_mean=("SystemTokens", "mean"),
        User_mean=("UserTokens", "mean"),
        Total_mean=("TotalPromptTokens", "mean"),
        Total_min=("TotalPromptTokens", "min"),
        Total_median=("TotalPromptTokens", "median"),
        Total_max=("TotalPromptTokens", "max"),
    )
    .reset_index()
)

round_cols = ["System_mean", "User_mean", "Total_mean", "Total_min", "Total_median", "Total_max"]
rq1_prompt_token_summary_df[round_cols] = rq1_prompt_token_summary_df[round_cols].round(1)

display(rq1_prompt_token_summary_df)

rq1_detail_path = OUTPUT_DIR / "rq1_prompt_token_counts_by_sample.xlsx"
rq1_summary_path = OUTPUT_DIR / "rq1_prompt_token_counts_summary.xlsx"
rq1_prompt_token_detail_df.to_excel(rq1_detail_path, index=False)
rq1_prompt_token_summary_df.to_excel(rq1_summary_path, index=False)

print(f"Saved RQ1 detail file: {rq1_detail_path}")
print(f"Saved RQ1 summary file: {rq1_summary_path}")
